Suspicious Fake News Detector
Dataset from Kaggle()

In [133]:
import pandas as pd
df =pd.read_csv('Fake.csv')
df.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [134]:
df.columns

Index(['title', 'text', 'subject', 'date'], dtype='object')

In [135]:
df1 = pd.read_csv('True.csv')
df1.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [136]:
df['label']=1
df1['label']=0

In [137]:
df =pd.concat([df, df1], axis =0)

In [138]:
df=df.sample(frac =1, random_state =1).reset_index(drop =True)
df.head(10)

,title,text,subject,date,label
0,Trump Calls For This Racist Policy To Be Forc...,Donald Trump is calling for one of the most co...,News,"September 21, 2016",1
1,Republican ex-defense secretary Cohen backs Hi...,WASHINGTON (Reuters) - Former Republican U.S. ...,politicsNews,"September 7, 2016",0
2,"TEACHER QUITS JOB After 5th, 6th Grade Muslim ...",You re never to young to commit jihad Teachers...,politics,"May 9, 2017",1
3,LAURA INGRAHAM RIPS INTO THE PRESS…Crowd Goes ...,Laura Ingraham reminds the Never Trump people ...,politics,"Jul 21, 2016",1
4,Germany's Merkel suffers state vote setback as...,BERLIN/HANOVER (Reuters) - Germany s Social De...,worldnews,"October 14, 2017",0
5,WHOA! MELANIA TRUMP BREAKS HER SILENCE…Fires B...,Melanie Trump is more than just a pretty face....,politics,"Oct 17, 2016",1
6,Study Shows Democrats Are Better Drivers For ...,You don t have to look beyond Washington to pr...,News,"February 1, 2017",1
7,U.S. lawmakers want moratorium on commercial f...,WASHINGTON (Reuters) - Two leading critics of ...,politicsNews,"September 7, 2016",0
8,The new risk for Europe: an inward-looking Ger...,"BERLIN (Reuters) - In 2008, in a fit of pique ...",worldnews,"September 26, 2017",0
9,"Trump still standing, but damaged by Comey's t...",WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"June 9, 2017",0


In [139]:
df.isna().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [140]:
import re

def remove_reuters(text):
    text = re.sub(r'^[A-Z/\s]+\(Reuters\)\s-', '', text)
    return text

df["text"] = df["text"].apply(remove_reuters)

In [141]:
df['content']=df['title'] +' '+ df['text']

In [142]:
df.duplicated().sum()

np.int64(209)

In [143]:
df =df.drop_duplicates(subset=['content'])
df.duplicated().sum()

np.int64(0)

In [144]:
df.head(10)

,title,text,subject,date,label,content
0,Trump Calls For This Racist Policy To Be Forc...,Donald Trump is calling for one of the most co...,News,"September 21, 2016",1,Trump Calls For This Racist Policy To Be Forc...
1,Republican ex-defense secretary Cohen backs Hi...,Former Republican U.S. Defense Secretary Will...,politicsNews,"September 7, 2016",0,Republican ex-defense secretary Cohen backs Hi...
2,"TEACHER QUITS JOB After 5th, 6th Grade Muslim ...",You re never to young to commit jihad Teachers...,politics,"May 9, 2017",1,"TEACHER QUITS JOB After 5th, 6th Grade Muslim ..."
3,LAURA INGRAHAM RIPS INTO THE PRESS…Crowd Goes ...,Laura Ingraham reminds the Never Trump people ...,politics,"Jul 21, 2016",1,LAURA INGRAHAM RIPS INTO THE PRESS…Crowd Goes ...
4,Germany's Merkel suffers state vote setback as...,Germany s Social Democrats (SPD) defeated Ang...,worldnews,"October 14, 2017",0,Germany's Merkel suffers state vote setback as...
5,WHOA! MELANIA TRUMP BREAKS HER SILENCE…Fires B...,Melanie Trump is more than just a pretty face....,politics,"Oct 17, 2016",1,WHOA! MELANIA TRUMP BREAKS HER SILENCE…Fires B...
6,Study Shows Democrats Are Better Drivers For ...,You don t have to look beyond Washington to pr...,News,"February 1, 2017",1,Study Shows Democrats Are Better Drivers For ...
7,U.S. lawmakers want moratorium on commercial f...,Two leading critics of President Barack Obama...,politicsNews,"September 7, 2016",0,U.S. lawmakers want moratorium on commercial f...
8,The new risk for Europe: an inward-looking Ger...,"In 2008, in a fit of pique over Angela Merkel...",worldnews,"September 26, 2017",0,The new risk for Europe: an inward-looking Ger...
9,"Trump still standing, but damaged by Comey's t...",President Donald Trump survived one of the bi...,politicsNews,"June 9, 2017",0,"Trump still standing, but damaged by Comey's t..."


In [145]:
from sklearn.model_selection import train_test_split
X = df['content']
y =df['label']
X_train, X_test, y_train, y_test =train_test_split(X, y, test_size =0.2, random_state =1, stratify =y)

In [146]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer =TfidfVectorizer( stop_words ='english', max_features =5000)
X_train_vec =vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [147]:
from sklearn.linear_model import LogisticRegression
model =LogisticRegression()
model.fit(X_train_vec, y_train)
y_pred =model.predict(X_test_vec)

In [148]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score
accuracy = accuracy_score(y_test, ypred)
precision =precision_score(y_test, ypred)
f1 =f1_score(y_test, ypred)
recall= recall_score(y_test, ypred)

In [149]:
print(f'Accuracy = {accuracy}')
print(f'Precision = {precision}')
print(f'Recall =  {recall}')
print(f'F1 = {f1}')

Accuracy = 0.980309423347398
Precision = 0.9855524079320114
Recall =  0.9712451144611949
F1 = 0.9783464566929134


In [150]:
df['label'].value_counts(normalize =True)

label
0    0.542053
1    0.457947
Name: proportion, dtype: float64

In [151]:
from sklearn.dummy import DummyClassifier
dummy =DummyClassifier(strategy = 'most_frequent')
dummy.fit(X_train_vec, y_train)
y_pred =dummy.predict(X_test_vec)

In [152]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score
accuracy = accuracy_score(y_test, ypred)
precision =precision_score(y_test, ypred)
f1 =f1_score(y_test, ypred)
recall= recall_score(y_test, ypred)
print(f'Accuracy = {accuracy}')
print(f'Precision = {precision}')
print(f'Recall =  {recall}')
print(f'F1 = {f1}')

Accuracy = 0.980309423347398
Precision = 0.9855524079320114
Recall =  0.9712451144611949
F1 = 0.9783464566929134
